In [1]:
import os
import pandas as pd
import numpy as np
import random
import optuna
import json

# Sembunyikan log Optuna agar terminal bersih
optuna.logging.set_verbosity(optuna.logging.WARNING)

TARGET_FOLDER = os.path.join('osmnx_inputs')

# ─────────────────────────────────────────────────────────────────────
# 1. FUNGSI FITNESS ACO 
# ─────────────────────────────────────────────────────────────────────
def hitung_fitness_aco_final(solusi_multi_hari, dist_np, elev_np, road_np, cluster_np, max_km_user):
    w_jarak = 10.0
    w_jalan = 5.0
    w_penalti_cluster = 50.0 
    w_hari = 30.0  
    
    fitness_total = len(solusi_multi_hari) * w_hari
    log_itinerary = []
    
    for idx_hari, rute in enumerate(solusi_multi_hari, start=1):
        if idx_hari % 2 != 0:
            nama_hari = f"Pekan {(idx_hari+1)//2} - Sabtu (Jalan Berat)"
            w_penalti_fatigue = 1.5; w_penalti_jarak = 5.0     
        else:
            nama_hari = f"Pekan {idx_hari//2} - Minggu (Recovery)"
            w_penalti_fatigue = 8.0; w_penalti_jarak = 25.0    
            
        jarak_m = fatigue = skor_jalan = p_cluster = 0.0
        
        for k in range(len(rute) - 1):
            a, t = rute[k], rute[k+1]
            
            dm = dist_np[a, t]
            dz = elev_np[a, t]
            js = road_np[a, t]
            
            jarak_m += dm
            skor_jalan += js
            
            if dz > 0 and dm > 0: 
                fatigue += dm * ((dz / dm) ** 2) * 400.0
            else: 
                fatigue += dm * 0.0001
                
            if a != 0 and t != 0:
                if cluster_np[a-1] != cluster_np[t-1]:
                    p_cluster += w_penalti_cluster

        jkm = jarak_m / 1000.0
        p_jarak = max(0.0, jkm - max_km_user) * w_penalti_jarak  
        p_fatigue = fatigue * w_penalti_fatigue
            
        fitness_hari = (jkm * w_jarak) + (skor_jalan * w_jalan) + p_jarak + p_fatigue + p_cluster
        fitness_total += fitness_hari
        
        log_itinerary.append({
            "Hari/Trip": nama_hari, "Jarak (Km)": round(jkm, 2),
            "Fatigue Index": round(fatigue, 2), "Skor Jalan OSMnx": round(skor_jalan, 2),
            "P_Fatigue": round(p_fatigue, 2), "P_Cluster": round(p_cluster, 2)
        })
        
    return fitness_total, log_itinerary

# ─────────────────────────────────────────────────────────────────────
# 2. CORE ENGINE ACO (DENGAN BLACKLIST NODE)
# ─────────────────────────────────────────────────────────────────────
# TAMBAHAN: Parameter banned_nodes dimasukkan ke engine
def run_aco_final(dist_np, elev_np, road_np, cluster_np, alpha, beta, evaporation, max_km_user, banned_nodes, num_ants=15, iterations=30):
    num_nodes = len(dist_np)
    max_m = max_km_user * 1000
    pheromone = np.full((num_nodes, num_nodes), 0.1)
    
    max_dist = dist_np.max()
    max_elev = elev_np.max() 
    if max_elev <= 0: max_elev = 1.0 
    
    best_score = float('inf')
    best_route = []
    best_log = None
    
    for _ in range(iterations):
        list_solusi_semut = []
        list_fitness_semut = []
        list_log_semut = []
        
        for ant in range(num_ants):
            # FILTER BLACKLIST: Semut hanya melihat destinasi yang tidak di-banned
            destinasi_tersisa = [i for i in range(1, num_nodes) if i not in banned_nodes]
            rute_multi_hari = []
            
            while len(destinasi_tersisa) > 0:
                rute_hari_ini = [0]
                jarak_hari = 0.0
                
                while len(destinasi_tersisa) > 0:
                    curr = rute_hari_ini[-1]
                    
                    valid_candidates = []
                    for cand in destinasi_tersisa:
                        uji_dist = dist_np[curr, cand]
                        jarak_kembali_hotel = dist_np[cand, 0]
                        if jarak_hari + uji_dist + jarak_kembali_hotel <= max_m:
                            valid_candidates.append(cand)
                            
                    if not valid_candidates:
                        break 
                        
                    probs = []
                    for cand in valid_candidates:
                        tau = pheromone[curr][cand] ** alpha
                        d_ij = dist_np[curr, cand]
                        road_ij = road_np[curr, cand]
                        dz = elev_np[curr, cand]
                        
                        d_norm = d_ij / max_dist
                        r_norm = road_ij / 5.0
                        
                        cluster_norm = 0.0
                        if curr != 0 and cand != 0:
                            if cluster_np[curr-1] != cluster_np[cand-1]: 
                                cluster_norm = 0.15 
                                
                        if len(rute_multi_hari) % 2 == 0:
                            eta = 1.0 / (0.8 * d_norm + 0.2 * r_norm + cluster_norm + 0.001)
                        else:
                            f_local = max(0, dz)
                            f_norm = f_local / (max_elev + 1e-6) 
                            eta = 1.0 / (0.5 * d_norm + 0.2 * r_norm + 0.3 * f_norm + cluster_norm + 0.001)
                            
                        probs.append(tau * (eta ** beta))
                        
                    sum_p = sum(probs)
                    if sum_p == 0: next_c = random.choice(valid_candidates)
                    else: next_c = random.choices(valid_candidates, weights=[p/sum_p for p in probs], k=1)[0]
                    
                    rute_hari_ini.append(next_c)
                    jarak_hari += dist_np[curr, next_c]
                    destinasi_tersisa.remove(next_c)
                    
                rute_hari_ini.append(0)
                
                if len(rute_hari_ini) == 2:
                    raise ValueError(f"Infeasible Error Lolos: Terdapat rute destinasi yang jarak PP-nya melebihi {max_km_user} km.")
                
                rute_multi_hari.append(rute_hari_ini)
                
            f_score, lg = hitung_fitness_aco_final(rute_multi_hari, dist_np, elev_np, road_np, cluster_np, max_km_user)
            list_solusi_semut.append(rute_multi_hari)
            list_fitness_semut.append(f_score)
            list_log_semut.append(lg)
            
            if f_score < best_score:
                best_score = f_score
                best_route = rute_multi_hari
                best_log = lg
                
        pheromone *= (1.0 - evaporation)
        for idx, solusi in enumerate(list_solusi_semut):
            fit_value = list_fitness_semut[idx]
            if fit_value > 0:
                deposit = 2000 / fit_value
                for rute in solusi:
                    for k in range(len(rute) - 1):
                        pheromone[rute[k]][rute[k+1]] += deposit
                        
    return best_score, best_route, best_log

# ─────────────────────────────────────────────────────────────────────
# 3. GENERATE DASHBOARD HTML
# ─────────────────────────────────────────────────────────────────────
def generate_dashboard(rute_terbaik, log_metrik, names, route_registry, target_folder):
    print("\n" + "=" * 65)
    print("🌍 MEMBUAT DASHBOARD PETA INTERAKTIF...")
    
    OUTPUT_DASHBOARD = os.path.join(target_folder, 'dashboard_interaktif_aco.html')
    
    data_metrik = []
    for log in log_metrik:
        data_metrik.append({
            "hari": log["Hari/Trip"],
            "jarak": log["Jarak (Km)"],
            "fatigue": log["Fatigue Index"],
            "jalan": log["Skor Jalan OSMnx"]
        })
        
    data_rute_js = []
    for rute in rute_terbaik:
        hari_data = []
        for k in range(len(rute) - 1):
            idx_asal = rute[k]
            idx_tujuan = rute[k+1]
            key = f"{idx_asal}_{idx_tujuan}"
            
            if key in route_registry:
                jalur_detail = route_registry[key]
                hari_data.append({
                    "asal_nama": names[idx_asal],
                    "tujuan_nama": names[idx_tujuan],
                    "is_depot_asal": (idx_asal == 0),
                    "is_depot_tujuan": (idx_tujuan == 0),
                    "jalur": jalur_detail 
                })
        data_rute_js.append(hari_data)

    json_rute_str = json.dumps(data_rute_js)
    json_metrik_str = json.dumps(data_metrik)

    html_content = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Dashboard Optimasi Rute Sepeda ACO - Surabaya</title>
        <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
        <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
        <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap" rel="stylesheet">
        
        <style>
            body {{ margin: 0; padding: 0; font-family: 'Inter', sans-serif; display: flex; height: 100vh; background-color: #f8f9fa; }}
            #map-container {{ flex: 1; height: 100%; position: relative; }}
            #map {{ height: 100%; width: 100%; }}
            .legend {{ position: absolute; bottom: 30px; left: 30px; z-index: 1000; background: white; padding: 15px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }}
            .legend-title {{ font-weight: bold; margin-bottom: 8px; font-size: 14px; }}
            .gradient-bar {{ width: 250px; height: 15px; border-radius: 4px; background: linear-gradient(to right, #27ae60, #f1c40f, #e74c3c); }}
            .legend-labels {{ display: flex; justify-content: space-between; margin-top: 5px; font-size: 12px; color: #555; }}
            #sidebar {{ width: 400px; height: 100%; background: white; box-shadow: -2px 0 10px rgba(0,0,0,0.1); overflow-y: auto; padding: 20px; box-sizing: border-box; z-index: 1000; }}
            h2 {{ margin-top: 0; color: #2c3e50; font-size: 20px; border-bottom: 2px solid #3498db; padding-bottom: 10px; }}
            .card {{ background: #ffffff; border: 1px solid #e0e6ed; border-radius: 8px; padding: 15px; margin-bottom: 15px; cursor: pointer; transition: all 0.3s ease; }}
            .card:hover {{ transform: translateY(-3px); box-shadow: 0 5px 15px rgba(0,0,0,0.08); border-color: #3498db; }}
            .card.active {{ border: 2px solid #3498db; background-color: #f0f7fb; }}
            .card-header {{ font-weight: 700; color: #2c3e50; margin-bottom: 10px; font-size: 15px; }}
            .badge {{ display: inline-block; padding: 3px 8px; border-radius: 12px; font-size: 11px; font-weight: bold; color: white; margin-bottom: 10px; }}
            .badge.sabtu {{ background-color: #e74c3c; }}
            .badge.minggu {{ background-color: #27ae60; }}
            .metric-row {{ display: flex; justify-content: space-between; margin-bottom: 5px; font-size: 13px; color: #555; }}
            .metric-val {{ font-weight: bold; color: #2c3e50; }}
            #btn-semua {{ width: 100%; padding: 12px; background: #34495e; color: white; border: none; border-radius: 6px; font-weight: bold; cursor: pointer; margin-bottom: 20px; transition: background 0.2s; }}
            #btn-semua:hover {{ background: #2c3e50; }}
        </style>
    </head>
    <body>
        <div id="map-container">
            <div id="map"></div>
            <div class="legend">
                <div class="legend-title">Elevasi Tanjakan Rute (m)</div>
                <div class="gradient-bar"></div>
                <div class="legend-labels">
                    <span>0m (Datar)</span>
                    <span>Landai</span>
                    <span>>25m (Berat)</span>
                </div>
            </div>
        </div>
        <div id="sidebar">
            <h2>Itinerary Multi-Hari ACO</h2>
            <button id="btn-semua" onclick="tampilkanSemuaRute()">Tampilkan Seluruh Rute</button>
            <div id="cards-container"></div>
        </div>
        <script>
            const dataRute = {json_rute_str};
            const dataMetrik = {json_metrik_str};
            
            const map = L.map('map').setView([-7.262015, 112.739727], 13);
            L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png', {{
                attribution: '&copy; OpenStreetMap contributors &copy; CARTO'
            }}).addTo(map);

            let layerGroup = L.layerGroup().addTo(map);

            function getColorForElevation(elev) {{
                const minE = 0, midE = 12.5, maxE = 25;
                let val = Math.max(minE, Math.min(elev, maxE)); 
                let r, g, b;
                if (val <= midE) {{
                    let ratio = val / midE;
                    r = Math.round(0x27 + ratio * (0xf1 - 0x27));
                    g = Math.round(0xae + ratio * (0xc4 - 0xae));
                    b = Math.round(0x60 + ratio * (0x0f - 0x60));
                }} else {{
                    let ratio = (val - midE) / (maxE - midE);
                    r = Math.round(0xf1 + ratio * (0xe7 - 0xf1));
                    g = Math.round(0xc4 + ratio * (0x4c - 0xc4));
                    b = Math.round(0x0f + ratio * (0x3c - 0x0f));
                }}
                return `rgb(${{r}}, ${{g}}, ${{b}})`;
            }}

            function gambarRuteKePeta(arrayIndexHari) {{
                layerGroup.clearLayers(); 
                let bounds = []; 
                let markersDitaruh = new Set();
                
                arrayIndexHari.forEach(idx_hari => {{
                    const ruteSatuHari = dataRute[idx_hari];
                    ruteSatuHari.forEach(segmen => {{
                        const polyline = segmen.jalur;
                        for (let i = 0; i < polyline.length - 1; i++) {{
                            let p1 = polyline[i];
                            let p2 = polyline[i+1];
                            let elevRata = (p1[2] + p2[2]) / 2;
                            let line = L.polyline([[p1[0], p1[1]], [p2[0], p2[1]]], {{
                                color: getColorForElevation(elevRata),
                                weight: 5,
                                opacity: 0.85
                            }}).addTo(layerGroup);
                            bounds.push([p1[0], p1[1]]);
                        }}
                        
                        const titikAwal = polyline[0];
                        const titikAkhir = polyline[polyline.length - 1];
                        
                        if (!markersDitaruh.has(segmen.asal_nama)) {{
                            let warnaIkon = segmen.is_depot_asal ? 'orange' : '#3498db';
                            L.circleMarker([titikAwal[0], titikAwal[1]], {{
                                radius: segmen.is_depot_asal ? 8 : 6,
                                color: 'white', weight: 2, fillColor: warnaIkon, fillOpacity: 1
                            }}).bindPopup(`<b>${{segmen.asal_nama}}</b><br>Elevasi: ${{titikAwal[2].toFixed(1)}}m`).addTo(layerGroup);
                            markersDitaruh.add(segmen.asal_nama);
                        }}
                        
                        if (!markersDitaruh.has(segmen.tujuan_nama)) {{
                            let warnaIkon = segmen.is_depot_tujuan ? 'orange' : '#3498db';
                            L.circleMarker([titikAkhir[0], titikAkhir[1]], {{
                                radius: segmen.is_depot_tujuan ? 8 : 6,
                                color: 'white', weight: 2, fillColor: warnaIkon, fillOpacity: 1
                            }}).bindPopup(`<b>${{segmen.tujuan_nama}}</b><br>Elevasi: ${{titikAkhir[2].toFixed(1)}}m`).addTo(layerGroup);
                            markersDitaruh.add(segmen.tujuan_nama);
                        }}
                    }});
                }});
                
                if(bounds.length > 0) map.fitBounds(bounds, {{padding: [30, 30]}});
            }}

            function tampilkanSemuaRute() {{
                document.querySelectorAll('.card').forEach(c => c.classList.remove('active'));
                let semuaIndex = dataMetrik.map((_, i) => i);
                gambarRuteKePeta(semuaIndex);
            }}

            const container = document.getElementById('cards-container');
            dataMetrik.forEach((metrik, idx) => {{
                let isSabtu = metrik.hari.toLowerCase().includes('sabtu');
                let badgeClass = isSabtu ? 'sabtu' : 'minggu';
                let badgeText = isSabtu ? 'HEAVY CLIMB' : 'RECOVERY';
                
                let card = document.createElement('div');
                card.className = 'card';
                card.innerHTML = `
                    <div class="badge ${{badgeClass}}">${{badgeText}}</div>
                    <div class="card-header">${{metrik.hari}}</div>
                    <div class="metric-row"><span>Total Jarak:</span> <span class="metric-val">${{metrik.jarak.toFixed(2)}} Km</span></div>
                    <div class="metric-row"><span>Fatigue Index:</span> <span class="metric-val">${{metrik.fatigue.toFixed(2)}}</span></div>
                    <div class="metric-row"><span>Skor Jalan:</span> <span class="metric-val">${{metrik.jalan.toFixed(2)}}</span></div>
                `;
                
                card.onclick = () => {{
                    document.querySelectorAll('.card').forEach(c => c.classList.remove('active'));
                    card.classList.add('active'); 
                    gambarRuteKePeta([idx]);
                }};
                
                container.appendChild(card);
            }});

            window.onload = tampilkanSemuaRute;
        </script>
    </body>
    </html>
    """

    with open(OUTPUT_DASHBOARD, "w", encoding="utf-8") as file:
        file.write(html_content)

    print(f"🎉 SELESAI! Buka file ini di browsermu: {OUTPUT_DASHBOARD}")

# ─────────────────────────────────────────────────────────────────────
# 4. MAIN EXECUTION & PRE-FLIGHT CHECK
# ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    try:
        matriks_jarak = pd.read_csv(os.path.join(TARGET_FOLDER, 'distance_matrix_osmnx.csv'), index_col=0)
        matriks_jalan = pd.read_csv(os.path.join(TARGET_FOLDER, 'road_condition_matrix_osmnx.csv'), index_col=0)
        matriks_elevasi = pd.read_csv(os.path.join(TARGET_FOLDER, 'elevation_matrix_surabaya.csv'), index_col=0)
        df_cluster = pd.read_csv(os.path.join(TARGET_FOLDER, 'destinasi_clustered_real_distance.csv'))
        
        ROUTE_JSON_PATH = os.path.join(TARGET_FOLDER, 'route_polyline_registry.json')
        with open(ROUTE_JSON_PATH, 'r') as f:
            route_registry = json.load(f)
            
        names = matriks_jarak.index.tolist()

        dist_np = matriks_jarak.to_numpy()
        road_np = matriks_jalan.to_numpy()
        elev_np = matriks_elevasi.to_numpy()
        cluster_np = df_cluster['Cluster_ID'].to_numpy()

        print("\n" + "=" * 65)
        print("🚲 PRE-FLIGHT SANITY CHECK: KAPASITAS USER")
        print("=" * 65)

        # INTERAKTIF: Meminta input batas jarak kepada user
        BANNED_NODES = []
        MAX_KM_USER = 20.10
        
        while True:
            try:
                input_user = input("\nMasukkan batas maksimal jarak per hari (Km) [Tekan Enter untuk default 20.10]: ")
                if input_user.strip():
                    MAX_KM_USER = float(input_user)
                else:
                    MAX_KM_USER = 25.0
            except ValueError:
                print("❌ Masukkan angka yang valid!")
                continue

            max_m = MAX_KM_USER * 1000
            infeasible_nodes = []

            # Validasi semua destinasi: Jarak PP (Pergi + Pulang)
            for i in range(1, len(names)):
                jarak_pp = dist_np[0, i] + dist_np[i, 0]
                if jarak_pp > max_m:
                    infeasible_nodes.append((i, names[i], jarak_pp))

            if infeasible_nodes:
                print(f"\n⚠️ WARNING: Ditemukan {len(infeasible_nodes)} destinasi yang jarak PP-nya saja sudah melebihi {MAX_KM_USER} km!")
                for idx, nama, jarak in infeasible_nodes:
                    print(f"   - {nama} (Jarak PP mutlak: {round(jarak/1000, 2)} km)")
                
                opsi = input("\nApakah kamu ingin MENGHAPUS destinasi di atas agar model tetap feasible? (y/n): ").strip().lower()
                if opsi == 'y':
                    print("✅ Destinasi infeasible di-blacklist dari pencarian rute.")
                    BANNED_NODES = [n[0] for n in infeasible_nodes]
                    break
                else:
                    print("❌ Pencarian dibatalkan. Silakan masukkan limit yang lebih besar untuk menjangkau destinasi tersebut.")
            else:
                BANNED_NODES = []
                print("✅ Semua destinasi lolos uji kapasitas fisik harian!")
                break

        print("\n" + "=" * 65)
        print("🐜 TUNING PARAMETER ACO FINAL DENGAN DATASET OSMNX LOKAL")
        print("=" * 65)

        def objective_final(trial):
            alpha = trial.suggest_float('alpha', 0.1, 1.5)
            beta = trial.suggest_float('beta', 2.0, 8.0)
            evaporation = trial.suggest_float('evaporation', 0.05, 0.5)
            
            # Memasukkan banned_nodes ke dalam run
            score, _, _ = run_aco_final(dist_np, elev_np, road_np, cluster_np, 
                                        alpha, beta, evaporation, max_km_user=MAX_KM_USER, banned_nodes=BANNED_NODES, num_ants=15, iterations=15)
            return score

        study = optuna.create_study(direction='minimize')
        study.optimize(objective_final, n_trials=20)
        best_p = study.best_params
        
        print(f"\n✅ Tuning Selesai! Skor Optuna Terbaik: {round(study.best_value, 2)}")
        print(f"✅ Parameter Terbaik: {best_p}")

        print(f"\n▶ Mengeksekusi Eksperimen Final ACO dengan Parameter Terbaik...")
        score_f, rute_f, log_f = run_aco_final(
            dist_np, elev_np, road_np, cluster_np,
            alpha=best_p['alpha'], beta=best_p['beta'], evaporation=best_p['evaporation'],
            max_km_user=MAX_KM_USER, banned_nodes=BANNED_NODES, num_ants=40, iterations=50
        )

        print("\n" + "=" * 65)
        print("====== HASIL OPTIMASI AKHIR MULTI-WEEKEND ITINERARY ACO ======")
        print("=" * 65)
        print(pd.DataFrame(log_f).to_string(index=False))
        print(f"\nSkor Akhir Fungsi Fitness: {round(score_f, 2)}")
        print(f"Total Waktu Liburan Keliling Surabaya: {len(rute_f)} Hari ({len(rute_f)//2} Akhir Pekan)")
        
        pd.DataFrame(log_f).to_csv(os.path.join(TARGET_FOLDER, 'itinerary_multi_weekend_aco.csv'), index=False)
        print(f"\nBerkas laporan skripsi berhasil diekspor ke folder: {TARGET_FOLDER}")
        
        generate_dashboard(rute_f, log_f, names, route_registry, TARGET_FOLDER)

    except FileNotFoundError as e:
        print(f"[ERROR] Berkas data preprocessing tidak ditemukan: {e}")
        print(f"Pastikan file csv & json sudah dipindahkan ke dalam folder: {TARGET_FOLDER}")


🚲 PRE-FLIGHT SANITY CHECK: KAPASITAS USER


/Users/mac/sc_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Semua destinasi lolos uji kapasitas fisik harian!

🐜 TUNING PARAMETER ACO FINAL DENGAN DATASET OSMNX LOKAL

✅ Tuning Selesai! Skor Optuna Terbaik: 19926.96
✅ Parameter Terbaik: {'alpha': 1.3303568959953482, 'beta': 6.951281314063239, 'evaporation': 0.33744060887925875}

▶ Mengeksekusi Eksperimen Final ACO dengan Parameter Terbaik...

====== HASIL OPTIMASI AKHIR MULTI-WEEKEND ITINERARY ACO ======
                    Hari/Trip  Jarak (Km)  Fatigue Index  Skor Jalan OSMnx  P_Fatigue  P_Cluster
Pekan 1 - Sabtu (Jalan Berat)       29.28        1042.15             74.14    1563.23        0.0
  Pekan 1 - Minggu (Recovery)       29.47         532.85             38.49    4262.76       50.0
Pekan 2 - Sabtu (Jalan Berat)       28.12        6612.24             27.76    9918.36        0.0
  Pekan 2 - Minggu (Recovery)       16.81         379.66              6.68    3037.29        0.0

Skor Akhir Fungsi Fitness: 20723.68
Total Waktu Liburan Keliling Surabaya: 4 Hari (2 Akhir Pekan)

Berkas laporan